## Generative Adversarial Networks (GANs) Tutorial

This notebook covers everything that I need to learn about generative ML with GAN technique based on the 2 materials provided: 
- https://www.tensorflow.org/hub/tutorials/tf_hub_generative_image_module
- https://www.tensorflow.org/tutorials/generative/dcgan

### Explain the roles of the generator and the discriminator in a GAN

#### What are GANs?

- a type of ML model that can generate new data (like images) that looks similar to training data
- they work by training **two neural networks** against each other in a "game":
  - **Generator**: Tries to create fake data that looks real
  - **Discriminator**: Tries to tell the difference between real and fake data

**Great analogy**: Think of it like a counterfeiter (generator) trying to fool an art expert (discriminator). As the expert gets better at spotting fakes, the counterfeiter must improve their technique. This competition makes both better over time!

#### The Generator

**Role**: Creates fake images from random noise

**How it works**:
- Takes a random vector (noise) as input
- Processes it through layers to transform noise into an image
- Tries to make images that look realistic
- Gets better when it successfully fools the discriminator

#### The Discriminator

**Role**: Classifies images as real or fake

**How it works**:
- Takes an image as input (either real from training data or fake from generator)
- Processes it through layers to extract features
- Outputs a single value: probability that the image is real
- Gets better at distinguishing real from fake images

#### How They Work Together

1. **Training Step**:
- Generator creates fake images from random noise
- Discriminator sees both real images (from dataset) and fake images (from generator)
- Discriminator tries to correctly identify which are real and which are fake
- Generator tries to make images that fool the discriminator

2. **Loss Functions**:
- Discriminator loss: How wrong it was on real + fake images (wants to minimize this)
- Generator loss: How well it fooled the discriminator (wants to maximize fooling, minimize this loss)

3. **Competition**:
- When discriminator gets good → generator must improve
- When generator gets good → discriminator must improve
- This creates a motivation that improves both networks -> similar to Minimax algorithm

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import matplotlib.pyplot as plt
import imageio

tf.random.set_seed(64)
np.random.seed(64)

In [ ]:
# Load the pre-trained Progressive GAN model
progan = hub.load("https://tfhub.dev/google/progan-128/1").signatures['default']

# The latent space dimension (random noise vector size)
LATENT_DIM = 512

In [ ]:
def generate_random_faces(num_faces):
    """Generate random faces from random noise vectors"""

    random_vectors = tf.random.normal([num_faces, LATENT_DIM])
    generated_images = progan(random_vectors)['default']
    
    # Convert from [-1, 1] range to [0, 1] for display
    images_np = generated_images.numpy()
    images_np = (images_np + 1.0) / 2.0
    images_np = np.clip(images_np, 0, 1)
    
    return images_np

# Generate 4 random faces
faces = generate_random_faces(num_faces=4)

# Display them
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    axes[i].imshow(faces[i])
    axes[i].set_title(f'Face {i+1}', fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
def interpolate_hypersphere(v1, v2, num_steps):
    """
    Interpolate between two latent vectors on a hypersphere.
    This creates smooth transitions between two faces.
    """
    v1_norm = tf.norm(v1)
    v2_norm = tf.norm(v2)
    v2_normalized = v2 * (v1_norm / v2_norm)
    
    vectors = []
    for step in range(num_steps):
        interpolated = v1 + (v2_normalized - v1) * step / (num_steps - 1)
        interpolated_norm = tf.norm(interpolated)
        interpolated_normalized = interpolated * (v1_norm / interpolated_norm)
        vectors.append(interpolated_normalized)
    
    return tf.stack(vectors)

v1 = tf.random.normal([LATENT_DIM])
v2 = tf.random.normal([LATENT_DIM])
interpolated_vectors = interpolate_hypersphere(v1, v2, num_steps=30)

# Generate images from interpolated vectors
generated_images = progan(interpolated_vectors)['default']

# Convert to displayable format
images_np = generated_images.numpy()
images_np = (images_np + 1.0) / 2.0
images_np = np.clip(images_np, 0, 1)

# Display a few frames from the morph
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
indices = [0, num_steps//4, num_steps//2, 3*num_steps//4, num_steps-1]
for i, idx in enumerate(indices):
    axes[i].imshow(images_np[idx])
    axes[i].set_title(f'Step {idx}', fontsize=10)
    axes[i].axis('off')
plt.tight_layout()
plt.show()

# Save as GIF 
converted_images = (images_np * 255).astype(np.uint8)
imageio.mimsave('face_morph.gif', converted_images, duration=0.1)


### Create an artwork by manipulating the latent space to generate unique facial features

Let's create unique artwork by exploring the latent space in different ways:
1. Facial feature variations
2. Latent space walks
3. Face blending
4. Grid interpolations


In [ ]:
# Method 1: Facial Feature Manipulation

# Start with a base face and modify it slightly to see feature changes
base_vector = tf.random.normal([LATENT_DIM])
base_face_vector = tf.expand_dims(base_vector, 0)
base_face = progan(base_face_vector)['default'][0]
base_face_np = (base_face.numpy() + 1.0) / 2.0
base_face_np = np.clip(base_face_np, 0, 1)

# Create variations by adding different amounts of noise
variations = []
variation_names = ['Base Face', 'Variation 1', 'Variation 2', 'Variation 3']

for i, strength in enumerate([0, 0.2, 0.3, 0.5]):
    if i == 0:
        variations.append(base_face_np)
    else:
        modified_vector = base_vector + tf.random.normal([LATENT_DIM]) * strength
        modified_face = progan(tf.expand_dims(modified_vector, 0))['default'][0]
        modified_face_np = (modified_face.numpy() + 1.0) / 2.0
        modified_face_np = np.clip(modified_face_np, 0, 1)
        variations.append(modified_face_np)

# Display
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (img, name) in enumerate(zip(variations, variation_names)):
    axes[i].imshow(img)
    axes[i].set_title(name, fontsize=12)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Method 2: Latent Space Walk

# Walk through latent space in a specific direction to see how features change
start_vector = tf.random.normal([1, LATENT_DIM])
direction = tf.random.normal([1, LATENT_DIM])
direction = direction / tf.norm(direction)

walk_images = []
for step in range(10):
    current_vector = start_vector + (direction * step * 0.1)
    generated = progan(current_vector)['default'][0]
    generated_np = (generated.numpy() + 1.0) / 2.0
    generated_np = np.clip(generated_np, 0, 1)
    walk_images.append(generated_np)

# Display
walk_array = np.array(walk_images)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flatten()):
    if i < len(walk_images):
        ax.imshow(walk_images[i])
        ax.set_title(f'Step {i}', fontsize=10)
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Method 3: Face Blending
# Blend multiple faces together by combining their latent vectors

# Generate 2 random seed faces
vectors = [tf.random.normal([LATENT_DIM]) for _ in range(2)]

# Create blend progression
blended_faces = []
for i in range(10):
    alpha = i / 9.0  # Blend factor from 0 to 1
    blended_vector = (1 - alpha) * vectors[0] + alpha * vectors[1]
    generated = progan(tf.expand_dims(blended_vector, 0))['default'][0]
    generated_np = (generated.numpy() + 1.0) / 2.0
    generated_np = np.clip(generated_np, 0, 1)
    blended_faces.append(generated_np)

# Display
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(blended_faces[i])
    ax.set_title(f'Blend {i*11}%', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Method 4: Interpolation Grid
# Create a grid showing interpolations between 4 corner faces

# Generate 4 corner faces
seed_vectors = [tf.random.normal([LATENT_DIM]) for _ in range(4)]

# Create interpolation grid
grid_size = 5
all_images = []

for i in range(grid_size):
    row_images = []
    for j in range(grid_size):
        # Bilinear interpolation between corners
        alpha = i / (grid_size - 1)
        beta = j / (grid_size - 1)
        
        top = (1 - beta) * seed_vectors[0] + beta * seed_vectors[1]
        bottom = (1 - beta) * seed_vectors[2] + beta * seed_vectors[3]
        interpolated = (1 - alpha) * top + alpha * bottom
        
        face = progan(tf.expand_dims(interpolated, 0))['default'][0]
        face_np = (face.numpy() + 1.0) / 2.0
        face_np = np.clip(face_np, 0, 1)
        row_images.append(face_np)
    
    all_images.append(np.hstack(row_images))

final_grid = np.vstack(all_images)

plt.figure(figsize=(10, 10))
plt.imshow(final_grid)
plt.axis('off')
plt.title('Latent Space Interpolation Grid', fontsize=16, pad=20)
plt.show()

### Create an artwork of any kind that fits your interests. You can use any GAN project on the internet or develop it yourself

What I'm going to do:
1. Use the ProGAN model to generate faces and manipulate them
2. Explore latent space in creative ways
3. Combine multiple techniques i've learned

In [28]:
# Create a spiral animation through latent space

center = tf.random.normal([LATENT_DIM])
spiral_images = []
num_frames = 40

for i in range(num_frames):
    angle = 2 * np.pi * i / num_frames
    radius = 0.6 * (i / num_frames)
    
    # Create spiral by rotating in a 2D subspace of latent space
    direction = tf.random.normal([LATENT_DIM])
    direction = direction / tf.norm(direction)
    
    # Create orthogonal direction
    orthogonal = tf.random.normal([LATENT_DIM])
    orthogonal = orthogonal - tf.reduce_sum(orthogonal * direction) * direction
    orthogonal = orthogonal / tf.norm(orthogonal)
    
    # Spiral motion
    spiral_vector = center + radius * (np.cos(angle) * direction + np.sin(angle) * orthogonal)
    face = progan(tf.expand_dims(spiral_vector, 0))['default'][0]
    face_np = (face.numpy() + 1.0) / 2.0
    face_np = np.clip(face_np, 0, 1)
    spiral_images.append((face_np * 255).astype(np.uint8))

# Save as GIF
imageio.mimsave('custom_artwork_spiral.gif', spiral_images, duration=0.15)

In [26]:
# Generate faces that transition through different "styles"

# Generate 5 different style seed vectors
style_vectors = [tf.random.normal([LATENT_DIM]) for _ in range(5)]

# Create smooth transitions between all styles in a loop
transition_frames = []
num_frames = 20

for i in range(len(style_vectors)):
    v1 = style_vectors[i]
    v2 = style_vectors[(i + 1) % len(style_vectors)]  # Loop back to first
    
    for frame in range(num_frames):
        alpha = frame / num_frames
        
        # Spherical interpolation for smooth transitions
        v1_norm = tf.norm(v1)
        v2_norm = tf.norm(v2)
        v2_n = v2 * (v1_norm / v2_norm)
        interpolated = v1 + alpha * (v2_n - v1)
        interpolated_norm = tf.norm(interpolated)
        interpolated = interpolated * (v1_norm / interpolated_norm)
        
        face = progan(tf.expand_dims(interpolated, 0))['default'][0]
        face_np = (face.numpy() + 1.0) / 2.0
        face_np = np.clip(face_np, 0, 1)
        transition_frames.append((face_np * 255).astype(np.uint8))

# Save as GIF
imageio.mimsave('custom_artwork_styles.gif', transition_frames, duration=0.1)